# Firmware Validation Walkthrough

Replays event logs to controllers with new firmware and compares output to the originals.

## Multi-Day Workflow

**First day (setup):** Run cells 1-5 to configure the suite and see which databases go on which controllers.

**Each batch day:**
1. Run the **"Select Current Batch"** cell — it auto-detects the next incomplete batch and shows which databases to load.
2. Load the listed databases onto the test controllers.
3. Run the **"Controller Check"** cell to verify connectivity.
4. Run the **"Run Replay"** cell — it replays only the selected batch, then checkpoints.
5. Come back the next day and repeat for the next batch.

**Final day (after all batches complete):** Run the **"Generate Report"** cell to produce the final comparison HTML report.


In [4]:
from pathlib import Path
import signal_replay as sr
print('signal_replay version:', sr.__version__)

signal_replay version: 0.1.0


## 2. Configure Paths and Targets

Edit `controller_targets` with controller targets in either format:
- `host:port` (expanded to same UDP+HTTP port), or
- `host:udp_port:http_port` (explicit mapping).

`batch_size` is set to however many controllers you have ? one TSSU per controller per batch.

In [5]:

from pathlib import Path

# This notebook lives in firmware_validation/, which also holds all the data
project_root  = Path('..').resolve()
firmware_dir  = Path('.').resolve()   # firmware_validation/ - logs, databases, results all live here

logs_dir      = firmware_dir / 'logs'
databases_dir = firmware_dir / 'databases'
catalog_path  = firmware_dir / 'databases.xlsx'

# --- Edit these to match your setup (host:udp_port:http_port) ---
controller_targets = [
    '127.0.0.1:1041',
    '127.0.0.1:1042',
    '127.0.0.1:1033',
    '127.0.0.1:1034',
    '127.0.0.1:1035',
    '127.0.0.1:1036',
    '127.0.0.1:1037',
    '127.0.0.1:1038',
    '127.0.0.1:1039',
]
# ----------------------------------------------------------------

batch_size       = len(controller_targets)
firmware_version = '2.15.1'

print(f'Project root:  {project_root}')
print(f'Firmware dir:  {firmware_dir}')
print(f'Batch size:    {batch_size}')
for t in controller_targets:
    print(f'  {t}')


Project root:  \\scdata2\signalshar\Data_Analysis\Python\ATC-Signal-Replay
Firmware dir:  \\scdata2\signalshar\Data_Analysis\Python\ATC-Signal-Replay\firmware_validation
Batch size:    9
  127.0.0.1:1041
  127.0.0.1:1042
  127.0.0.1:1033
  127.0.0.1:1034
  127.0.0.1:1035
  127.0.0.1:1036
  127.0.0.1:1037
  127.0.0.1:1038
  127.0.0.1:1039


## 3. Read the Excel Catalog

Reads `firmware_validation/databases.xlsx`. Expected columns: `TSSU`, `Version`, `Type`, `CycleLength`, `Offset`, `Notes`.
`CycleLength` and `Offset` are optional — when present, they configure coordinated-signal replay timing instead of time-of-day alignment.
Rows are processed in order, top-to-bottom.


In [6]:

from openpyxl import load_workbook

wb     = load_workbook(catalog_path, read_only=True, data_only=True)
ws     = wb.active
rows   = list(ws.iter_rows(values_only=True))
wb.close()

header  = [str(h).strip().lower() for h in rows[0]]
catalog = []
for row in rows[1:]:
    if not any(row):   # skip blank rows
        continue
    r = dict(zip(header, [str(v).strip() if v is not None else '' for v in row]))
    
    # Parse optional numeric columns (CycleLength, Offset)
    raw_cl = r.get('cyclelength', '')
    raw_off = r.get('offset', '')
    cycle_length = int(float(raw_cl)) if raw_cl not in ('', 'None') else 0
    offset = float(raw_off) if raw_off not in ('', 'None') else 0.0
    
    catalog.append({
        'TSSU':         r.get('tssu', ''),
        'Version':      r.get('version', ''),
        'Type':         r.get('type', 'Similarity') or 'Similarity',
        'CycleLength':  cycle_length,
        'Offset':       offset,
        'Notes':        r.get('notes', ''),
    })

print(f'{len(catalog)} rows loaded from databases.xlsx\n')
for i, r in enumerate(catalog, 1):
    cl_str = f'  CL={r["CycleLength"]}' if r['CycleLength'] else ''
    off_str = f'  Off={r["Offset"]}' if r['Offset'] else ''
    print(f"  {i:3d}. {r['TSSU']:>8s}  {r['Type']:<12s}  {r['Version']}{cl_str}{off_str}")


26 rows loaded from databases.xlsx

    1.    03013  Similarity    2.14.0
    2.    05018  Similarity    2.14.0
    3.    08042  Similarity    2.15.1
    4.    08404  Similarity    2.15.1
    5.    08411  Similarity    2.15.1
    6.    12035  Similarity    2.14.0
    7.    12036  Similarity    2.14.0
    8.    12059  Similarity    2.14.0
    9.    01066  Similarity    2.14.0
   10.    13008  Similarity    2.14.0
   11.    13010  Similarity    2.14.0
   12.    2B009  Similarity    2.9.2
   13.    2B045  Similarity    2.8.0
   14.    2B049  Similarity    2.14.0
   15.    2B052  Similarity    2.8.0
   16.    2B054  Similarity    2.8.0
   17.    2B085  Similarity    2.8.0
   18.    2B094  Similarity    2.8.0
   19.    2B337  Similarity    2.9.2
   20.    2B339  Similarity    2.9.2
   21.    2B349  Similarity    2.8.0
   22.    2B530  Similarity    2.12.0
   23.    2C009  Similarity    2.2.0
   24.    2C042  Similarity    2.9.2
   25.    2C043  Similarity    2.14.0
   26.  2B045_c  Conflict

## 4. Check File Readiness

Shows which log files (`.parquet`, `.csv`, `.db`) and database files exist for each TSSU. Missing logs are skipped during the run.

In [7]:
def find_log(tssu):
    """Look for TSSU.parquet, TSSU.csv, TSSU.db, or any TSSU* match."""
    for ext in ('.parquet', '.csv', '.db'):
        p = logs_dir / f'{tssu}{ext}'
        if p.exists():
            return p
    matches = sorted(logs_dir.glob(f'{tssu}*'))
    return matches[0] if matches else None

def find_database(tssu):
    """Look for any file starting with TSSU in databases_dir."""
    matches = sorted(databases_dir.glob(f'{tssu}*'))
    return matches[0] if matches else None

file_map = {}
ready_count = 0
for r in catalog:
    tssu = r['TSSU']
    log = find_log(tssu)
    db  = find_database(tssu)
    file_map[tssu] = {'log': log, 'db': db}
    ready_count += bool(log)
    log_name = log.name if log else 'MISSING'
    db_name  = db.name  if db  else '--'
    status   = 'OK' if log else '--'
    print(f'  {status:>2s}  {tssu:>8s}  log={log_name:<30s}  db={db_name}')

print(f'\nReady: {ready_count} / {len(catalog)}')


  OK     03013  log=03013.parquet                   db=03013.bin
  OK     05018  log=05018.parquet                   db=05018.bin
  OK     08042  log=08042.parquet                   db=08042.bin
  OK     08404  log=08404.parquet                   db=08404.bin
  OK     08411  log=08411.parquet                   db=08411.bin
  OK     12035  log=12035.parquet                   db=12035.bin
  OK     12036  log=12036.parquet                   db=12036.bin
  OK     12059  log=12059.parquet                   db=12059.bin
  OK     01066  log=01066.parquet                   db=01066.bin
  OK     13008  log=13008.parquet                   db=13008.bin
  OK     13010  log=13010.parquet                   db=13010.bin
  OK     2B009  log=2B009.parquet                   db=2B009.bin
  OK     2B045  log=2B045.parquet                   db=2B045.bin
  OK     2B049  log=2B049.parquet                   db=2B049.bin
  OK     2B052  log=2B052.parquet                   db=2B052.bin
  OK     2B054  log=2B054

## 5. Build Test Suite and Batches

Creates scenarios for rows that have log files, splits them into batches of `batch_size`,
and prints a **load table** for each batch showing exactly which database file goes on which controller IP:port.

### 5a. Configure Conflict Pairs (per device)
For **Conflict** test types, specify which phase pairs should be monitored for overlaps. 
If a TSSU is not listed in `conflict_pairs`, no conflict monitoring will occur for that device.

In [8]:
import json

# Load per-device conflict pairs from JSON
# Format: 'TSSU_ID': [['phaseA', 'phaseB'], ...]
conflict_pairs_path = firmware_dir / 'conflict_monitor' / 'conflict_pairs.json'

if conflict_pairs_path.exists():
    with open(conflict_pairs_path, 'r') as f:
        conflict_pairs = json.load(f)
        
    # Convert lists to tuples for compatibility with the rest of the code
    conflict_pairs = {k: [tuple(pair) for pair in v] for k, v in conflict_pairs.items()}
else:
    conflict_pairs = {}

if conflict_pairs:
    print('Configured conflict pairs:')
    for tssu, pairs in conflict_pairs.items():
        print(f'  {tssu}: {pairs}')
else:
    print('No conflict pairs configured. Monitoring will be disabled for all devices.')


Configured conflict pairs:
  12059: [('Ph1', 'Ph2'), ('Ph1', 'Ph4'), ('Ph1', 'Ph8'), ('Ph1', 'O1'), ('Ph1', 'O2'), ('Ph1', 'O4'), ('Ph1', 'Ped2'), ('Ph1', 'Ped4'), ('Ph1', 'Ped6'), ('Ph1', 'Ped8'), ('Ph2', 'Ph4'), ('Ph2', 'Ph8'), ('Ph2', 'O3'), ('Ph2', 'Ped4'), ('Ph2', 'Ped8'), ('Ph4', 'Ph5'), ('Ph4', 'Ph6'), ('Ph4', 'Ph8'), ('Ph4', 'O1'), ('Ph4', 'O4'), ('Ph4', 'Ped2'), ('Ph4', 'Ped6'), ('Ph4', 'Ped8'), ('Ph5', 'Ph6'), ('Ph5', 'Ph8'), ('Ph5', 'Ped4'), ('Ph5', 'Ped6'), ('Ph5', 'Ped8'), ('Ph6', 'Ph8'), ('Ph6', 'O4'), ('Ph6', 'Ped4'), ('Ph6', 'Ped8'), ('Ph8', 'O2'), ('Ph8', 'O3'), ('Ph8', 'O4'), ('Ph8', 'Ped2'), ('Ph8', 'Ped4'), ('Ph8', 'Ped6'), ('O1', 'O3'), ('O1', 'O4'), ('O1', 'Ped4'), ('O2', 'Ped8'), ('O3', 'O4'), ('O3', 'Ped2'), ('O3', 'Ped8'), ('O4', 'Ped4'), ('O4', 'Ped8'), ('Ped2', 'Ped4'), ('Ped2', 'Ped8'), ('Ped4', 'Ped6'), ('Ped4', 'Ped8'), ('Ped6', 'Ped8')]
  12036: [('Ph2', 'Ph8'), ('Ph5', 'Ph6'), ('Ph5', 'Ph8'), ('Ph5', 'O1'), ('Ph5', 'Ped6'), ('Ph6', 'Ph8'), ('Ph8', 'O1'),

In [9]:

scenarios = []
skipped   = []
for r in catalog:
    tssu = r['TSSU']
    log  = file_map[tssu]['log']
    db   = file_map[tssu]['db']

    if not log:
        skipped.append(tssu)
        continue

    test_type = sr.TestType.CONFLICT if r['Type'].strip().lower() == 'conflict' else sr.TestType.SIMILARITY

    # Read cycle_length and offset from catalog (0 = default / disabled)
    cycle_length = r.get('CycleLength', 0) or 0
    offset       = r.get('Offset', 0.0) or 0.0

    # tod_align must be False when cycle_length > 0 (they are incompatible)
    tod_align = cycle_length == 0

    kwargs = {}
    if test_type == sr.TestType.CONFLICT:
        kwargs['replays'] = 25
        # Try explicit key first, then fall back to base TSSU (strip _c suffix)
        pairs_key = tssu if tssu in conflict_pairs else tssu.rstrip('_c') if tssu.endswith('_c') else tssu
        if pairs_key in conflict_pairs:
            kwargs['incompatible_pairs'] = conflict_pairs[pairs_key]

    scenarios.append(sr.TestScenario(
        scenario_id    = tssu,
        database_name  = str(db) if db else f'{tssu}.bin',
        events_source  = str(log),
        test_type      = test_type,
        description    = f"{r['Type']} | {r['Notes']}" if r['Notes'] else r['Type'],
        cycle_length   = cycle_length,
        cycle_offset   = offset,
        tod_align      = tod_align,
        **kwargs,
    ))

if skipped:
    print(f'Skipped (no log): {", ".join(skipped)}')

def normalize_target(target: str) -> str:
    """Normalize target to host:udp_port:http_port."""
    parts = target.split(':')
    if len(parts) == 2:
        host, port_text = parts
        port = int(port_text)
        return f'{host}:{port}:{port}'
    if len(parts) == 3:
        host, udp_text, http_text = parts
        return f'{host}:{int(udp_text)}:{int(http_text)}'
    raise ValueError(
        f'Invalid controller target: {target}. Use host:port or host:udp_port:http_port.'
    )

normalized_targets = [normalize_target(t) for t in controller_targets]

# Chunk into batches of batch_size
batches = []
for i in range(0, len(scenarios), batch_size):
    chunk     = scenarios[i:i + batch_size]
    batch_num = i // batch_size + 1
    targets   = normalized_targets[:len(chunk)]
    batches.append(sr.TestBatch(
        batch_id    = f'batch_{batch_num}',
        assignments = {s.scenario_id: t for s, t in zip(chunk, targets)},
        description = f'Batch {batch_num}: {len(chunk)} scenarios',
    ))

suite = sr.FirmwareTestSuite(
    suite_name   = 'Firmware Validation',
    firmware_version  = firmware_version,
    baseline_version  = 'original_logs',
    scenarios    = scenarios,
    batches      = batches,
    output_dir   = str(firmware_dir / 'results'),
    comparison_thresholds = sr.ComparisonThresholds(
        sequence_threshold = 0.05,
        timing_threshold   = 0.02,
        match_threshold    = 95.0,
    ),
)

yaml_path = firmware_dir / 'test_suite.yaml'
sr.save_to_yaml(suite, str(yaml_path))

# Summary + DB -> Controller load instructions
print(f'Scenarios: {len(scenarios)}  |  Batches: {len(batches)}\n')
for b in batches:
    print(f'-- {b.batch_id}: load these databases before starting --')
    for sid, ip in b.assignments.items():
        s      = next(s for s in scenarios if s.scenario_id == sid)
        # Extract just the filename for cleaner display
        db_name = Path(s.database_name).name

        extras = []
        if s.test_type == sr.TestType.CONFLICT:
            if s.incompatible_pairs:
                extras.append(f'pairs=configured')
            else:
                extras.append('(no conflict pairs set)')
        if s.cycle_length:
            extras.append(f'CL={s.cycle_length}')
        if s.cycle_offset:
            extras.append(f'Off={s.cycle_offset}')
        if not s.tod_align:
            extras.append('tod_align=OFF')

        extra_str = '  ' + ', '.join(extras) if extras else ''
        print(f'  {ip:<22s}  <-  {db_name:<20s}{extra_str}')
    print('-' * 70)
    print()

print(f'Saved: {yaml_path}')


Scenarios: 26  |  Batches: 3

-- batch_1: load these databases before starting --
  127.0.0.1:1041:1041     <-  03013.bin           
  127.0.0.1:1042:1042     <-  05018.bin           
  127.0.0.1:1033:1033     <-  08042.bin           
  127.0.0.1:1034:1034     <-  08404.bin           
  127.0.0.1:1035:1035     <-  08411.bin           
  127.0.0.1:1036:1036     <-  12035.bin           
  127.0.0.1:1037:1037     <-  12036.bin           
  127.0.0.1:1038:1038     <-  12059.bin           
  127.0.0.1:1039:1039     <-  01066.bin           
----------------------------------------------------------------------

-- batch_2: load these databases before starting --
  127.0.0.1:1041:1041     <-  13008.bin           
  127.0.0.1:1042:1042     <-  13010.bin           
  127.0.0.1:1033:1033     <-  2B009.bin           
  127.0.0.1:1034:1034     <-  2B045.bin           
  127.0.0.1:1035:1035     <-  2B049.bin           
  127.0.0.1:1036:1036     <-  2B052.bin           
  127.0.0.1:1037:1037     <- 

## 6. Select Current Batch

Detects the next incomplete batch from the checkpoint file.
Shows which databases to load on which controllers before proceeding.
If all batches are already complete, skip ahead to the report.


In [8]:
import json

# --- Detect next incomplete batch from checkpoint ---
checkpoint_path = Path(suite.output_dir) / firmware_version / 'checkpoint.json'
if checkpoint_path.exists():
    with open(checkpoint_path) as f:
        _ck = json.load(f)
    _completed = set(_ck.get('completed_batches', []))
else:
    _completed = set()

current_batch = None
for b in batches:
    if b.batch_id not in _completed:
        current_batch = b
        break

if current_batch is None:
    print('All batches are complete!  Skip ahead to the Report cell.')
else:
    print(f'Current batch:  {current_batch.batch_id}  ({len(current_batch.assignments)} scenarios)')
    print(f'Completed so far: {sorted(_completed) if _completed else "(none)"}\n')

    # Show what to load
    print('Load these databases onto the controllers before running replay:')
    for sid, tgt in current_batch.assignments.items():
        s = next(s for s in scenarios if s.scenario_id == sid)
        print(f'  {tgt:<22s}  <-  {Path(s.database_name).name}')

Current batch:  batch_3  (9 scenarios)
Completed so far: ['batch_1', 'batch_2']

Load these databases onto the controllers before running replay:
  127.0.0.1:1041:1041     <-  2B094.bin
  127.0.0.1:1042:1042     <-  2B337.bin
  127.0.0.1:1033:1033     <-  2B339.bin
  127.0.0.1:1034:1034     <-  2B349.bin
  127.0.0.1:1035:1035     <-  2B530.bin
  127.0.0.1:1036:1036     <-  2C009.bin
  127.0.0.1:1037:1037     <-  2C042.bin
  127.0.0.1:1038:1038     <-  2C043.bin
  127.0.0.1:1039:1039     <-  2B045_c.bin


### Controller Check

Polls SNMP + HTTP on each controller for the current batch until all respond OK.
Interrupt the cell (stop button) once you see all green, or let it auto-break.


In [10]:
import time
import requests
from concurrent.futures import ThreadPoolExecutor
from signal_replay.ntcip import send_ntcip

def parse_target(target):
    parts = target.split(':')
    if len(parts) == 2:
        ip, port_text = parts
        port = int(port_text)
        return ip, port, port
    if len(parts) == 3:
        ip, udp_text, http_text = parts
        return ip, int(udp_text), int(http_text)
    raise ValueError(f'Invalid target format: {target}')

def check_controller(target):
    ip, udp_port, http_port = parse_target(target)
    try:
        send_ntcip((ip, udp_port), 1, 1, 'Vehicle', timeout=1.0)
        send_ntcip((ip, udp_port), 1, 0, 'Vehicle', timeout=1.0)
        snmp_ok = True
    except Exception:
        snmp_ok = False
    try:
        r = requests.get(f'http://{ip}:{http_port}/v1/asclog/xml/full', timeout=2.0, verify=False)
        http_ok = (r.status_code == 200)
    except Exception:
        http_ok = False
    return snmp_ok and http_ok

GREEN_CHECK = '\033[92mOK\033[0m'
RED_X       = '\033[91mFAIL\033[0m'

if current_batch is None:
    print('All batches complete — nothing to check.')
else:
    db_lookup  = {s.scenario_id: Path(s.database_name).name for s in scenarios}
    b_targets = list(current_batch.assignments.values())
    b_labels  = [f'{tgt}/{db_lookup[sid]}' for sid, tgt in current_batch.assignments.items()]

    try:
        while True:
            with ThreadPoolExecutor() as ex:
                statuses = list(ex.map(check_controller, b_targets))
            parts = [GREEN_CHECK if ok else f'{RED_X} {lbl}' for ok, lbl in zip(statuses, b_labels)]
            print('\r\033[K' + '  '.join(parts), end='', flush=True)
            if all(statuses):
                print()
                break
            time.sleep(3)
    except KeyboardInterrupt:
        print()


OK  OK  OK  OK  FAIL 127.0.0.1:1035:1035/2B530.bin  OK  OK  OK  OKin 127.0.0.1:1039:1039/2B045_c.bin 127.0.0.1:1039:1039/2B045_c.bin


### Run Replay (Current Batch Only)

Runs replay for the current batch. The checkpoint is saved automatically so next time you come back, the batch-select cell will advance to the next one.


In [11]:
if current_batch is None:
    print('Nothing to run — all batches already complete.')
else:
    runner = sr.BatchRunner(suite, debug=False)

    def auto_db_loader(db_name, target):
        print(f'  [auto] DB {db_name} -> {target}')
        return True

    checkpoint = runner.run(
        db_loader_callback=auto_db_loader,
        batch_ids=[current_batch.batch_id],
    )
    print(f'\nCompleted batches: {checkpoint.get("completed_batches", [])}')

    remaining = [b.batch_id for b in batches if b.batch_id not in checkpoint.get('completed_batches', [])]
    if remaining:
        print(f'Remaining batches: {remaining}')
        print('Come back tomorrow, re-run from the "Select Current Batch" cell for the next batch.')
    else:
        print('All batches complete! Run the Report cell next.')

  [auto] DB \\scdata2\signalshar\Data_Analysis\Python\ATC-Signal-Replay\firmware_validation\databases\2B094.bin -> 127.0.0.1:1041:1041
  [auto] DB \\scdata2\signalshar\Data_Analysis\Python\ATC-Signal-Replay\firmware_validation\databases\2B337.bin -> 127.0.0.1:1042:1042
  [auto] DB \\scdata2\signalshar\Data_Analysis\Python\ATC-Signal-Replay\firmware_validation\databases\2B339.bin -> 127.0.0.1:1033:1033
  [auto] DB \\scdata2\signalshar\Data_Analysis\Python\ATC-Signal-Replay\firmware_validation\databases\2B349.bin -> 127.0.0.1:1034:1034
  [auto] DB \\scdata2\signalshar\Data_Analysis\Python\ATC-Signal-Replay\firmware_validation\databases\2B530.bin -> 127.0.0.1:1035:1035
  [auto] DB \\scdata2\signalshar\Data_Analysis\Python\ATC-Signal-Replay\firmware_validation\databases\2C009.bin -> 127.0.0.1:1036:1036
  [auto] DB \\scdata2\signalshar\Data_Analysis\Python\ATC-Signal-Replay\firmware_validation\databases\2C042.bin -> 127.0.0.1:1037:1037
  [auto] DB \\scdata2\signalshar\Data_Analysis\Python\A

2026-03-03 08:59:37,862 INFO Starting similarity batch batch_3 with 8 scenarios


  Input events stored in 4.6s
Starting ATC simulation with 8 signals, 1 replays
Replay data spans ~22h 59m (TOD-align: events sent at real wall-clock times)

--- Starting Run 1/1 ---
Sending events to 8 controllers...
[2C042] TOD align: date_shift=15d, skipping 0/67124 old events, first event at 09:00:02
[2C042] Waiting 22s until first event...
[2B337] TOD align: date_shift=15d, skipping 0/59574 old events, first event at 09:00:01
[2B337] Waiting 21s until first event...
[2C009] TOD align: date_shift=15d, skipping 0/267926 old events, first event at 09:00:00
[2C009] Waiting 19s until first event...
[2B094] TOD align: date_shift=15d, skipping 0/245987 old events, first event at 09:00:00
[2B094] Waiting 20s until first event...
[2C043] TOD align: date_shift=15d, skipping 0/163671 old events, first event at 09:00:00
[2C043] Waiting 19s until first event...
[2B339] TOD align: date_shift=15d, skipping 0/190688 old events, first event at 09:00:03
[2B339] Waiting 22s until first event...
[2B3

Datetime warning from 127.0.0.1:1036 (10001 events): Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Sample raw values: [Timestamp('2026-03-03 20:26:58.100000'), Timestamp('2026-03-03 20:26:58.300000'), Timestamp('2026-03-03 20:26:58.700000'), Timestamp('2026-03-03 20:26:58.900000'), Timestamp('2026-03-03 20:27:00.100000')]
Datetime warning from 127.0.0.1:1037 (10001 events): Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Sample raw values: [Timestamp('2026-03-03 20:26:29.300000'), Timestamp('2026-03-03 20:26:29.300000'), Timestamp('2026-03-03 20:26:29.300000'), Timestamp('2026-03-03 20:26:33.500000'), Timestamp('2026-03-03 20:26:33.600000')]
Datetime warning from 127.0.0.1:1037 (10001 events): Could not infer format, so each element will 

[2B337] Complete — sent 59574 events
[2C043] Complete — sent 163671 events
[2B094] Complete — sent 245987 events
[2C009] Complete — sent 267926 events
[2B349] Complete — sent 271134 events
[2B339] Complete — sent 190688 events
[2C042] Complete — sent 67124 events
[2B530] Complete — sent 249857 events


...

Run 1 completed

SIMULATION COMPLETE

Completed Runs: 1
Conflicts Found: 0
  [auto] DB \\scdata2\signalshar\Data_Analysis\Python\ATC-Signal-Replay\firmware_validation\databases\2B045_c.bin -> 127.0.0.1:1039:1039


2026-03-04 08:00:26,364 INFO Starting conflict scenario 2B045_c


  Input events stored in 0.3s
Starting ATC simulation with 1 signals, 25 replays
Estimated duration per run: 0h 32m (computed in 0.0s)

--- Starting Run 1/25 ---
Sending events to 1 controllers...
*** Signal 2B045_c failed: MIB module 'c:\Users\hwyr67g\AppData\Local\Programs\Python\Python311\Lib\site-packages\pysnmp\smi\mibs\SNMPv2-MIBc:\Users\hwyr67g\AppData\Local\Programs\Python\Python311\Lib\site-packages\pysnmp\smi\mibs\SNMPv2-MIB.py' load error: ['Traceback (most recent call last):\n', '  File "\\\\scdata2\\signalshar\\Data_Analysis\\Python\\ATC-Signal-Replay\\src\\signal_replay\\replay.py", line 548, in run\n    # No event loop running, safe to use asyncio.run\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^\n', 'RuntimeError: no running event loop\n', '\nDuring handling of the above exception, another exception occurred:\n\n', 'Traceback (most recent call last):\n', '  File "c:\\Users\\hwyr67g\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\pysnmp\\smi\\builder.py", line

2026-03-04 08:00:26,609 INFO Completed conflict scenario 2B045_c
2026-03-04 08:00:26,655 INFO Batch batch_3 complete



--- Skipping comparison (collection failed) ---

SIMULATION COMPLETE

Completed Runs: 0
Conflicts Found: 0
Signal Failures by Run:
  Run 1: 2B045_c

Completed batches: ['batch_1', 'batch_2', 'batch_3']
All batches complete! Run the Report cell next.


## 7. Generate Final Report

Run this **after all batches are complete**. It compares collected output against the original logs,
generates divergence charts, per-phase breakdowns, and produces the HTML report.


In [ ]:
import json, duckdb
import signal_replay as sr
from signal_replay.report import generate_report

# --- Load checkpoint to locate output databases ---
checkpoint_path = Path(suite.output_dir) / firmware_version / 'checkpoint.json'
with open(checkpoint_path) as f:
    cp = json.load(f)

completed = cp.get('completed_batches', [])
total_batches = [b.batch_id for b in suite.batches]
if set(total_batches) - set(completed):
    missing = sorted(set(total_batches) - set(completed))
    print(f'⚠️  Not all batches are complete. Missing: {missing}')
    print('   The report will only cover completed batches.')

plots_dir = Path(suite.output_dir) / firmware_version / 'divergence_plots'
plots_dir.mkdir(parents=True, exist_ok=True)

# --- Run comparisons sequentially ---
jobs = []
for scenario in suite.scenarios:
    db_path = cp['scenario_db_map'].get(scenario.scenario_id)
    if not db_path:
        print(f'  No output for {scenario.scenario_id}, skipping')
        continue
    jobs.append((scenario.scenario_id, scenario.events_source, scenario.test_type, db_path))

results = []
import pandas as pd

for sid, esrc, test_type, dbp in jobs:
    try:
        # Load original events
        original = sr.load_events(esrc)

        # Load collected events from DuckDB
        con = duckdb.connect(dbp)
        collected = con.execute(
            "SELECT * FROM events WHERE device_id = ? ORDER BY timestamp",
            [sid],
        ).df()
        con.close()

        # Compare runs
        result = sr.compare_runs(
            events_a=original, events_b=collected,
            device_id=sid,
            run_a_label='original_logs', run_b_label=firmware_version,
            auto_align=True, settle_minutes=3.0,
        )

        # Generate divergence charts + per-phase analysis
        plot_paths = []
        timeline_a = timeline_b = None
        if not original.empty and not collected.empty:
            try:
                timeline_a = sr.generate_timeline(original, device_id=sid)
                timeline_b = sr.generate_timeline(collected, device_id=sid)
                remove_events = ['Ped Omit', 'Phase Hold', 'Phase Omit', 'Phase Call',
                                 'Transition Longway', 'Transition Shortway']
                timeline_a = timeline_a[~timeline_a['EventClass'].isin(remove_events)]
                timeline_b = timeline_b[~timeline_b['EventClass'].isin(remove_events)]
            except Exception as e:
                print(f'    Timeline generation failed for {sid}: {e}')

        if result.divergence_windows and timeline_a is not None and not timeline_a.empty and not timeline_b.empty:
            try:
                time_offset_b = sr.compute_timeline_offset(timeline_a, timeline_b)
                plot_paths = sr.create_multi_divergence_plots(
                    timeline_a=timeline_a, timeline_b=timeline_b,
                    comparison_result=result, output_dir=str(plots_dir),
                    label_a='Original', label_b=firmware_version,
                    max_plots=3, window_minutes=5.0, time_offset_b=time_offset_b,
                )
            except Exception as e:
                print(f'    Chart generation failed for {sid}: {e}')

        passed = result.match_percentage >= 95.0
        phase_diffs = []
        if not passed and timeline_a is not None and not timeline_a.empty and not timeline_b.empty:
            try:
                settle_td = pd.Timedelta(minutes=3)
                tl_a_settled = timeline_a[timeline_a['StartTime'] >= timeline_a['StartTime'].min() + settle_td].copy()
                tl_b_settled = timeline_b[timeline_b['StartTime'] >= timeline_b['StartTime'].min() + settle_td].copy()
                overlap_end = min(tl_a_settled['EndTime'].max(), tl_b_settled['EndTime'].max())
                tl_a_settled = tl_a_settled[tl_a_settled['StartTime'] <= overlap_end]
                tl_b_settled = tl_b_settled[tl_b_settled['StartTime'] <= overlap_end]
                phase_diffs = sr.generate_phase_difference_summary(tl_a_settled, tl_b_settled, tolerance_seconds=0.2)
            except Exception as e:
                print(f'    Phase breakdown failed for {sid}: {e}')

        # Create result object
        results.append(sr.ScenarioResult(
            scenario_id=sid, test_type=test_type,
            firmware_version=firmware_version, passed=passed,
            match_percentage=result.match_percentage,
            num_divergences=len(result.divergence_windows),
            notes=result.format_summary(),
            plot_paths=plot_paths, phase_differences=phase_diffs,
            runs_completed=1, total_runs=1,
        ))
        
        status = "PASS" if passed else "FAIL"
        plots_msg = f", {len(plot_paths)} charts" if plot_paths else ""
        diffs_msg = ""
        if phase_diffs:
            diffs_msg = f"\n    Phase/overlap differences ({len(phase_diffs)} phases):\n"
            diffs_msg += sr.format_phase_differences(phase_diffs, label_a='Original', label_b=firmware_version)
        print(f'  {sid}: {result.match_percentage:.1f}%  {status}  ({len(result.divergence_windows)} divergences{plots_msg}){diffs_msg}')
        
    except Exception as e:
        import traceback
        print(f'  {sid}: ERROR — {e}')
        traceback.print_exc()

# Sort results to match suite order
order = {s.scenario_id: i for i, s in enumerate(suite.scenarios)}
results.sort(key=lambda r: order.get(r.scenario_id, 999))

print(f'\nResults: {sum(1 for r in results if r.passed)}/{len(results)} passed')

# --- Generate HTML report ---
report_path = Path(suite.output_dir) / 'report.html'
generate_report(results, suite, str(report_path))
print(f'\nReport saved to: {report_path}')
print(f'Total images embedded: {sum(len(r.plot_paths) for r in results)}')

: 

## 8. Archive Logs & Extract New Baseline

After reviewing the report, run this cell to:
1. Rename the current `logs/` folder to `logs_{firmware_version}/` as an archive
2. Create a fresh `logs/` folder
3. Extract collected events from DuckDB as parquet files — these become the new baseline for the next firmware version

In [ ]:
import shutil

logs_dir = Path('logs')
archive_dir = Path(f'logs_{firmware_version}')

if archive_dir.exists():
    print(f'Archive folder already exists: {archive_dir}')
    print('Skipping archive step — delete or rename it manually if you want to re-archive.')
else:
    if logs_dir.exists():
        logs_dir.rename(archive_dir)
        print(f'Archived logs/ -> {archive_dir}')
    else:
        print('No logs/ folder found — nothing to archive.')

# Create fresh logs/ for the next firmware version
logs_dir.mkdir(parents=True, exist_ok=True)
print(f'Fresh logs/ directory ready.')

# --- Extract collected events from DuckDB as parquet (new baseline) ---
checkpoint_path = Path(suite.output_dir) / firmware_version / 'checkpoint.json'
with open(checkpoint_path) as f:
    cp = json.load(f)

extracted = 0
for scenario in suite.scenarios:
    db_path = cp['scenario_db_map'].get(scenario.scenario_id)
    if not db_path:
        continue
    con = duckdb.connect(db_path)
    try:
        df = con.execute(
            "SELECT * FROM events WHERE device_id = ? ORDER BY timestamp",
            [scenario.scenario_id],
        ).df()
    finally:
        con.close()
    if df.empty:
        continue
    out_path = logs_dir / f'{scenario.scenario_id}.parquet'
    df.to_parquet(out_path, index=False)
    extracted += 1
    print(f'  {scenario.scenario_id} -> {out_path}  ({len(df)} events)')

print(f'\nExtracted {extracted} scenario(s) into logs/')